# MCP 서버 둘러보기 — 기억 · 검색 · RAG · 시장 데이터

MCP 서버 네 개를 차례로 붙여 봅니다. **코드는 거의 같고 서버만 바뀝니다.**
이것이 MCP의 요점입니다 — 도구를 만드는 대신 **이미 있는 서버를 꽂습니다.**

| Step | 서버 | 무엇을 해 주나 | 실행 |
|---|---|---|---|
| 2 | `server-memory` | 대화를 **지식 그래프로 기억** | `npx` |
| 3 | `tavily-mcp` | **웹 검색** | `npx` |
| 4 | `mcp-server-qdrant` | **벡터 DB** 저장·검색 (Agentic RAG) | `uvx` |
| 5 | `mcp_massive` | **주식 시장 데이터** | `uvx` |

네 서버 모두 `MCPServerStdio` 로 띄웁니다. 바뀌는 것은 `command` 와 `args` 뿐입니다.

> **준비물**: `npx`(Node.js), `uvx`(Python).
> `.env` 에 `OPENAI_API_KEY`, `TAVILY_API_KEY`, `MASSIVE_API_KEY` 가 필요합니다.

---

## Step 0. 준비

In [1]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio, create_static_tool_filter
import os
from pathlib import Path
from datetime import datetime
from IPython.display import Markdown, display

load_dotenv(override=True)

True

## Step 1. 주피터에서 꼭 필요한 손질

MCP의 stdio 서버는 **자식 프로세스**로 뜹니다. 노트북에서 그냥 연결하면 실패합니다.

```
io.UnsupportedOperation: fileno
```

MCP 라이브러리는 서버의 오류 출력을 `sys.stderr` 로 보내는데,
**주피터의 `sys.stderr` 는 진짜 파일이 아니라** 결과창으로 보내는 가짜 객체라
파일 번호(`fileno`)가 없습니다. 그래서 오류 출력을 **진짜 파일**로 돌려 줍니다.

> 서버가 뜨지 않을 때는 `mcp_server_errors.log` 를 열어 보세요.
> 버리지(`DEVNULL`) 않고 남겨 두는 이유입니다.

In [2]:
import functools

import agents.mcp.server

# 주피터의 sys.stderr 에는 파일 번호가 없어 하위 프로세스에 넘길 수 없다.
# 서버의 오류 출력을 진짜 파일로 돌려 둔다. (DEVNULL 로 버리면 원인을 못 본다)
MCP_LOG = open("mcp_server_errors.log", "w", encoding="utf-8")
agents.mcp.server.stdio_client = functools.partial(
    agents.mcp.server.stdio_client, errlog=MCP_LOG)

print("MCP 서버 오류 로그 -> mcp_server_errors.log")

MCP 서버 오류 로그 -> mcp_server_errors.log


## Step 2. 기억하는 MCP 서버 (server-memory)

LLM은 기본적으로 **지난 대화를 기억하지 못합니다.** 매번 처음부터입니다.
`server-memory` 는 대화에서 알게 된 사실을 **지식 그래프**로 파일에 적어 둡니다.

```
홍길동 ──[담당]──> LLM 교육 ──[포함]──> RAG, Fine-tuning, ...
 (개체)   (관계)      (개체)
```

| 환경변수 | 뜻 |
|---|---|
| `MEMORY_FILE_PATH` | 기억을 저장할 파일 경로 |

도구 이름을 보면 `create_entities`(개체 만들기) · `create_relations`(관계 잇기) ·
`search_nodes`(찾기) 처럼 **그래프를 다루는 동작**이 그대로 도구가 되어 있습니다.

In [3]:
# 기억을 보관할 파일. 서버가 이 파일에 지식 그래프를 저장한다.
memory_path = os.path.abspath("memory/memory.json")
os.makedirs(os.path.dirname(memory_path), exist_ok=True)

memory_params = {
    "command": "npx",
    "args": ["-y", "@modelcontextprotocol/server-memory"],
    "env": {"MEMORY_FILE_PATH": memory_path},
}

async with MCPServerStdio(params=memory_params, client_session_timeout_seconds=60) as server:
    memory_tools = await server.list_tools()

[tool.name for tool in memory_tools]

['create_entities',
 'create_relations',
 'add_observations',
 'delete_entities',
 'delete_observations',
 'delete_relations',
 'read_graph',
 'search_nodes',
 'open_nodes']

### 모델과 역할 정하기

지시문은 짧습니다. **무엇을 기억할지, 어떻게 저장할지는 지시하지 않습니다.**
에이전트가 도구 설명을 보고 알아서 판단합니다.

In [4]:
system_prompt = "당신은 long term 메모리를 유지하는 임무를 담당하고 있습니다."
model = "gpt-5.4-mini"

### (1) 기억시키기

이름과 담당 업무를 알려 줍니다. 에이전트는 이 사실을 **파일에 적습니다.**

In [5]:
async with MCPServerStdio(params=memory_params, client_session_timeout_seconds=60) as mcp_server:
    agent = Agent(name="agent", 
                  instructions=system_prompt, 
                  model=model, 
                  mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, "내 이름은 홍길동이고 이번에 LLM에 대한 교육을 담당하고 있습니다.  LLM 기본, RAG, Fine-tuning, Agentic AI에 대해 강의를 합니다")
    display(Markdown(result.final_output))

기억해둘게요:  
- 이름: 홍길동  
- 역할: LLM 교육 담당  
- 강의 주제: LLM 기본, RAG, Fine-tuning, Agentic AI

### (2) 다른 실행에서 떠올리기 — 여기가 핵심입니다

**위 셀과 완전히 별개의 실행입니다.** `async with` 블록이 끝나면서 서버도 에이전트도 사라졌고,
대화 기록도 넘기지 않았습니다. 그런데도 답을 합니다.

| | 앞 노트북들의 `checkpointer` | 이 방식 |
|---|---|---|
| 저장 위치 | 프로세스 메모리 | **파일** |
| 재시작하면 | 사라짐 | **남아 있음** |
| 저장 형태 | 대화 기록 그대로 | **정리된 사실(그래프)** |

대화를 통째로 쌓는 것이 아니라 **사실만 추려 저장**하므로, 대화가 길어져도 커지지 않습니다.
12장의 컨텍스트 엔지니어링이 서버 하나로 해결된 사례입니다.

> `memory/memory.json` 을 직접 열어 보세요. 무엇이 어떻게 저장됐는지 보입니다.

In [6]:
async with MCPServerStdio(params=memory_params, client_session_timeout_seconds=60) as mcp_server:
    agent = Agent(name="agent", 
                  instructions=system_prompt, 
                  model=model, 
                  mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, "내 이름은 홍길동입니다. LLM 교육에서 무엇을 강의하나요?")
    display(Markdown(result.final_output))

LLM 교육에서는 보통 이런 내용을 강의합니다:

- LLM의 기본 개념: 토큰, 임베딩, 트랜스포머
- 프롬프트 엔지니어링
- RAG(검색 증강 생성)
- 파인튜닝 / LoRA
- 평가 방법
- 안전성, 환각, 편향
- 실제 서비스 적용 사례

원하시면  
1) 초급용 커리큘럼,  
2) 기업 교육용 목차,  
3) 1일 강의안  
중 하나로 정리해드릴게요.

### 실행 기록 보기

`with trace("이름")` 으로 묶으면 LLM 호출과 **MCP 도구 호출까지** 하나의 기록이 됩니다.
어떤 도구를 어떤 인자로 불렀는지 전부 남으므로, "왜 이렇게 답했지?" 를 추적할 때 씁니다.

<https://platform.openai.com/traces>


---

## Step 3. 웹 검색 MCP (Tavily)

모델의 지식에는 **학습 시점(cutoff)** 이 있습니다. 그 이후 일은 모릅니다.
웹 검색 MCP 서버를 붙이면 **지금 정보**로 답할 수 있습니다.

| 환경변수 | 뜻 |
|---|---|
| `TAVILY_API_KEY` | Tavily 검색 API 키 (`.env` 에 넣어 둡니다) |

> 서버는 내 PC에서 돌지만 **리소스는 인터넷**입니다.
> "로컬 서버라서 안전하다"가 성립하지 않는 전형적인 예입니다.

In [7]:
tavily_params = {"command": "npx", "args": ["-y", "tavily-mcp@latest"], "env": {"TAVILY_API_KEY": os.getenv("TAVILY_API_KEY")}}

async with MCPServerStdio(params=tavily_params, client_session_timeout_seconds=60) as server:
    tavily_tools = await server.list_tools()

tavily_tools

[Tool(name='tavily_search', title=None, description='Search the web for current information on any topic. Use for news, facts, or data beyond your knowledge cutoff. Returns snippets and source URLs.', input_schema={'type': 'object', 'properties': {'query': {'type': 'string', 'description': 'Search query'}, 'search_depth': {'type': 'string', 'enum': ['basic', 'advanced', 'fast', 'ultra-fast'], 'description': "The depth of the search. 'basic' for generic results, 'advanced' for more thorough search, 'fast' for optimized low latency with high relevance, 'ultra-fast' for prioritizing latency above all else", 'default': 'basic'}, 'topic': {'type': 'string', 'enum': ['general'], 'description': 'The category of the search. This will determine which of our agents will be used for the search', 'default': 'general'}, 'time_range': {'type': 'string', 'description': 'The time range back from the current date to include in the search results', 'enum': ['day', 'week', 'month', 'year']}, 'start_date'

### 도구 필터 — 필요한 것만 노출하기

Tavily 서버는 검색 외에도 여러 도구를 제공합니다. 그중 **하나만** 쓰게 제한합니다.

```python
create_static_tool_filter(allowed_tool_names=["tavily_search"])
```

왜 막는가:

- **잘못 고를 여지를 줄인다** — 도구가 많을수록 엉뚱한 선택이 늘어납니다
- **토큰을 아낀다** — 도구 설명은 매 호출마다 모델에게 전송됩니다
- **권한을 좁힌다** — 쓰지 않을 기능은 아예 주지 않는 편이 안전합니다

15장의 최소 권한 원칙을 MCP 수준에서 적용하는 방법입니다.

In [8]:
system_prompt = "당신은 웹서치를 하고 요약하는 임무를 수행합니다."
model = "gpt-5.4-mini"
search_only = create_static_tool_filter(allowed_tool_names=["tavily_search"])

필터가 실제로 어떤 모양인지 확인합니다.

In [9]:
print(search_only)

{'allowed_tool_names': ['tavily_search']}


### 검색시키기

"어느 사이트를 보라"고 지정하지 않았습니다. 에이전트가 **검색어를 스스로 만들어** 찾습니다.

In [10]:
async with MCPServerStdio(params=tavily_params, client_session_timeout_seconds=60, tool_filter=search_only) as mcp_server:
    agent = Agent(name="agent", 
                  instructions=system_prompt, 
                  model=model, 
                  mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, "사이버보안 취약점 정보 포탈에 접속하여 가장 최근에 발생한 보안 취약점에 대해서 요약해 주세요")
    display(Markdown(result.final_output))

사이버 보안 취약점 정보 포털(KNVD) 기준, 가장 최근 등록된 취약점은 2026-10-15 기준의 **CVE-2026-79832**입니다.

- **제목:** HPE NonStop SQL/MX Executor - Buffer overflow vulnerability
- **제품:** HPE NonStop SQL/MX Executor
- **위험도:** **7.6 (High)**
- **핵심 요약:** 버퍼 오버플로 취약점으로, 악용 시 서비스 장애나 임의 코드 실행으로 이어질 수 있는 고위험 유형입니다.

같은 날짜에 함께 등록된 주요 취약점도 여러 건 있으며, 예를 들면:
- CVE-2026-79829: HPE NonStop HTTP WEBSERVER 메모리 손상
- CVE-2026-79828: HPE NonStop Debug Utilities 힙 버퍼 오버플로
- CVE-2026-18659: Jetty의 인증 우회 취약점

원하시면 제가 이 **최신 취약점만 더 자세히** 정리해드리겠습니다.

---

## Step 4. Agentic RAG — 벡터 DB MCP (Qdrant)

2권(RAG)에서는 청킹·임베딩·벡터 DB를 **직접** 만들었습니다.
여기서는 그 전부를 **서버 하나가** 대신합니다.

| 환경변수 | 뜻 |
|---|---|
| `QDRANT_LOCAL_PATH` | 벡터 DB를 저장할 폴더 (파일 모드 — 서버를 따로 띄울 필요 없음) |
| `COLLECTION_NAME` | 컬렉션 이름 |

도구는 둘뿐입니다.

| 도구 | 하는 일 |
|---|---|
| `qdrant-store` | 글을 **임베딩해서 저장** |
| `qdrant-find` | 질문과 **의미가 가까운** 글 찾기 |

임베딩 모델 선택, 청킹, 유사도 계산이 전부 서버 안에 숨어 있습니다.
**빠르게 만들 수 있는 대신, 세부를 조절할 수 없습니다** — 2권에서 직접 만들어 본 이유입니다.

In [11]:
vectordb_path = Path("memory/qdrant")
vectorstore_params = {
    "command": "uvx",
    "args": ["mcp-server-qdrant"],
    "env": {
        "QDRANT_LOCAL_PATH": str(vectordb_path),
        "COLLECTION_NAME": "knowledge",
    },
}

async with MCPServerStdio(params=vectorstore_params, client_session_timeout_seconds=120) as server:
    vectorstore_tools = await server.list_tools()

vectorstore_tools

[Tool(name='qdrant-find', title=None, description='Look up memories in Qdrant. Use this tool when you need to: \n - Find memories by their content \n - Access memories for further analysis \n - Get some personal information about the user', input_schema={'properties': {'query': {'description': 'What to search for', 'title': 'Query', 'type': 'string'}}, 'required': ['query'], 'type': 'object'}, execution=None, output_schema=None, icons=None, annotations=None, meta=None),
 Tool(name='qdrant-store', title=None, description='Keep the memory for later use, when you are asked to remember something.', input_schema={'properties': {'information': {'description': 'Text to store', 'title': 'Information', 'type': 'string'}, 'metadata': {'anyOf': [{'additionalProperties': True, 'type': 'object'}, {'type': 'null'}], 'default': None, 'description': 'Extra metadata stored along with memorised information. Any json is accepted.', 'title': 'Metadata'}}, 'required': ['information'], 'type': 'object'}, ex

### 서버 두 개를 함께 — 검색해서 저장하기

`mcp_servers=[search_server, vector_server]` 로 **두 서버를 동시에** 붙입니다.
에이전트는 Tavily 로 찾고 Qdrant 에 넣습니다. **그 순서를 우리가 짜지 않았습니다.**

이것이 *Agentic* RAG 입니다 — 무엇을 검색하고 무엇을 저장할지 에이전트가 정합니다.

In [12]:
system_prompt = """당신의 임무는 주어진 주제에 대해 웹서치를 하고 취득한 정보를 
               벡터 데이터베이스에 저장하는 것입니다. 사용자가 저장된 정보에 대해서 질문하면
               답변해 주세요"""

model = "gpt-5.4-mini"

async with MCPServerStdio(params=tavily_params, client_session_timeout_seconds=60, tool_filter=search_only) as search_server:
    async with MCPServerStdio(params=vectorstore_params, client_session_timeout_seconds=120) as vector_server:
        agent = Agent(name="researcher", 
                      instructions=system_prompt, 
                      model=model, 
                      mcp_servers=[search_server, vector_server])
        with trace("research and store"):
            result = await Runner.run(agent, "인공지능을 이용한 보안 취약성 공격에 대한 최근 뉴스를 서치하고 정보를 벡터 데이터베이스에 저장해 주세요", max_turns=20)
        display(Markdown(result.final_output))

저장했습니다.

요약:
- 최근 AI를 이용한 보안 취약성 공격 관련 뉴스로, 한국 금융권 7개 금융사에서 연관 공격 정황이 확인됨.
- 공격자는 AI 도구로 대규모 자동화 정찰·침투 시도를 했을 가능성이 제기됨.
- 피해는 주로 정보 유출 중심으로 보고됨.
- 대응 측면에서는 제로트러스트, MFA, 공격면 관리, 위협정보 공유, AI 기반 탐지/대응이 중요합니다.

이 내용으로 나중에 질문하시면 이어서 답변드릴게요.

### 저장한 것을 꺼내 쓰기

이번에는 **벡터 DB 서버만** 붙였습니다. 검색 서버는 없습니다.
그런데도 답할 수 있다면, 앞 셀에서 **실제로 저장이 되었다는 뜻**입니다.

> 답이 비어 있다면 앞 셀이 저장에 실패한 것입니다. `memory/qdrant` 폴더를 확인해 보세요.

In [13]:
async with MCPServerStdio(params=vectorstore_params, client_session_timeout_seconds=120) as vector_server:
    agent = Agent(name="researcher", 
                  instructions=system_prompt, 
                  model=model, 
                  mcp_servers=[vector_server])
    with trace("retrieve"):
        result = await Runner.run(agent, "인공지능을 이용한 최근 보안 공격은 어떤 내용이 있는지 벡터 데이터베이스는 내용에서 알려주세요")
    display(Markdown(result.final_output))

벡터 DB에 저장된 내용 기준으로 보면, 최근 인공지능을 이용한 보안 공격은 다음과 같습니다.

- **한국 금융권 대상 AI 활용 자동화 공격 의심 사례**
- **7개 금융사**(신한은행, KB국민은행, 하나은행, BNK부산은행, 예가람저축은행, 웰컴저축은행, 현대캐피탈)에서 **유사 공격 정황** 확인
- 공격자가 **AI 도구로 대규모 자동화 정찰·침투**를 시도했을 가능성
- **직원용/보조 시스템**, **외부 노출 서비스** 같은 상대적으로 약한 공격면이 진입점이었을 가능성
- 피해는 **금전 피해보다 정보 유출 중심**
- 시사점은 **AI가 공격 규모와 속도를 키울 수 있다**는 점입니다

원하시면 제가 이 내용을 더 짧게 요약하거나, **최근 AI 기반 보안 공격 유형별로 정리**해드릴게요.

---

## Step 5. 시장 데이터 MCP

주가·거래량처럼 **실시간으로 바뀌는 데이터**는 모델이 알 수 없습니다.
전문 데이터 제공자의 MCP 서버를 붙입니다.

```python
"args": ["--with", "mcp<2.0.0", "--from", "git+https://github.com/...", "mcp_massive"]
```

| 부분 | 뜻 |
|---|---|
| `--from git+https://...` | PyPI 가 아니라 **GitHub 저장소에서** 받아 실행 |
| `@v0.10.0` | 특정 태그로 고정 — 업데이트로 깨지지 않게 |
| `--with "mcp<2.0.0"` | 이 서버가 쓰는 MCP 라이브러리를 1.x 로 고정 |

> **버전 고정이 왜 필요한가**: MCP 파이썬 SDK가 2.x 로 올라가며 내부가 바뀌었습니다.
> 아직 1.x 전제로 만들어진 서버가 많습니다. MCP는 규격과 라이브러리가 **빠르게 바뀌는 단계**입니다.

도구 목록을 보면 `search_endpoints` 가 먼저입니다 —
**어떤 API 를 쓸지 찾고 나서** 호출하는 2단계 구조입니다. 도구가 수백 개일 때 쓰는 방식입니다.

In [14]:
massive_api_key = os.getenv("MASSIVE_API_KEY")

if not massive_api_key:
    raise RuntimeError(
        ".env 에 MASSIVE_API_KEY 가 없습니다. "
        "https://massive.com 에서 키를 발급받아 넣고 다시 실행하세요.")

market_params = {
    "command": "uvx",
    "args": ["--with", "mcp<2.0.0",
             "--from", "git+https://github.com/massive-com/mcp_massive@v0.10.0",
             "mcp_massive"],
    "env": {"MASSIVE_API_KEY": massive_api_key},
}

async with MCPServerStdio(params=market_params, client_session_timeout_seconds=120) as server:
    market_tools = await server.list_tools()

[tool.name for tool in market_tools]

['search_endpoints', 'call_api', 'query_data']

### 질문하기

"어떤 API 를 쓰라"고 알려 주지 않았습니다. 에이전트가 `search_endpoints` 로 찾아서 호출합니다.

In [25]:
system_prompt = "당신은 주식시장에 대한 보고를 합니다."
model = "gpt-5.4-mini"

async with MCPServerStdio(params=market_params, client_session_timeout_seconds=120) as mcp_server:
    agent = Agent(name="agent", instructions=system_prompt, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, "S&P 주식시장에서 최근 주식거래량이 가장 많은 Top5와 거래량을 알려주세요?")
    display(Markdown(result.final_output))

최근 거래량 Top 5는 아래와 같습니다.

1. **SNYR** — **1,083,075,453**
2. **DLXY** — **246,531,794**
3. **CTNT** — **239,718,630**
4. **QCLS** — **237,700,528**
5. **PDSB** — **196,516,305**

원하시면 **S&P 500 종목만 따로** 추려서 다시 보여드릴게요.

---

## 정리

### 네 서버를 같은 방식으로 붙였습니다

| | memory | tavily | qdrant | massive |
|---|---|---|---|---|
| 실행 | `npx` | `npx` | `uvx` | `uvx` (GitHub) |
| 리소스 | **내 파일** | 인터넷 | **내 파일** | 외부 API |
| 설정 | `MEMORY_FILE_PATH` | `TAVILY_API_KEY` | `QDRANT_LOCAL_PATH` | `MASSIVE_API_KEY` |
| 바뀐 코드 | `command`, `args`, `env` 뿐 | 〃 | 〃 | 〃 |

**`MCPServerStdio` 사용법은 네 번 모두 똑같았습니다.**
에이전트 쪽 코드도 `mcp_servers=[...]` 한 줄뿐입니다.
서버를 바꿔 끼우는 것만으로 에이전트가 할 수 있는 일이 달라집니다.

### 기억할 것

| | |
|---|---|
| 도구를 만들지 않는다 | 이미 있는 서버를 꽂습니다 |
| 서버 위치 ≠ 리소스 위치 | tavily 서버는 로컬이지만 인터넷을 봅니다 |
| 파일 기반 기억 | 프로세스가 죽어도 남습니다 |
| 도구 필터 | 쓰지 않을 도구는 아예 주지 않습니다 |
| 버전 고정 | MCP는 아직 빠르게 바뀝니다. `mcp<2` 같은 고정이 필요합니다 |
| 편한 만큼 가려진다 | Qdrant 서버는 청킹·임베딩을 숨깁니다. 조절하려면 직접 만들어야 합니다 |

### 더 해볼 것

1. `memory/memory.json` 을 열어 보세요. 그래프가 어떻게 생겼는지 보입니다.
2. Step 3 의 `tool_filter` 를 빼고 실행해 보세요. 에이전트가 다른 도구를 고르는지 관찰해 보세요.
3. Step 4 에서 저장 없이 꺼내기만 해 보세요(새 `COLLECTION_NAME` 으로). 빈 결과가 나옵니다.
4. 네 서버를 **한 에이전트에 모두** 붙여 보세요. 도구가 몇 개가 되는지, 선택이 흔들리는지 확인해 보세요.

---

## 직접 해보기

위 **더 해볼 것** 네 가지를 실행해 볼 수 있게 셀을 준비했습니다.
Step 0 ~ Step 5 의 셀을 먼저 실행해 두어야 합니다(`memory_params`, `tavily_params`, `vectorstore_params`, `market_params` 를 그대로 씁니다).

`# ✏️` 표시가 있는 줄을 바꿔 가며 실행해 보세요.

### 준비 — 에이전트가 부른 도구 보기

답만 봐서는 에이전트가 **어떤 도구를 골랐는지** 알 수 없습니다.
`result.new_items` 에서 도구 호출만 골라 출력하는 함수를 만들어 둡니다.

In [ ]:
from agents.items import ToolCallItem

def show_tool_calls(result):
    """에이전트가 실행 중에 부른 도구 이름과 인자를 순서대로 출력한다."""
    calls = [item.raw_item for item in result.new_items if isinstance(item, ToolCallItem)]
    print(f"도구 호출 {len(calls)}번")
    for i, call in enumerate(calls, 1):
        name = getattr(call, "name", type(call).__name__)
        args = getattr(call, "arguments", "")
        print(f"  {i}. {name}  {args[:120]}")

### 실습 1. 기억 파일 열어 보기

Step 2 에서 저장한 `memory/memory.json` 을 읽어 **개체(entity)** 와 **관계(relation)** 로 나눠 봅니다.
이 파일은 한 줄에 JSON 하나씩 들어 있는 형식(JSON Lines)입니다.

**관찰할 것**
- 홍길동은 어떤 `entityType` 으로 저장됐나요?
- `observations` 에는 문장이 그대로 들어갔나요, 요약됐나요?
- 관계 이름(`relationType`)은 누가 정했나요?

In [ ]:
import json

entities, relations = [], []
with open(memory_path, encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        record = json.loads(line)
        (entities if record.get("type") == "entity" else relations).append(record)

print(f"개체 {len(entities)}개")
for e in entities:
    print(f"  [{e['entityType']}] {e['name']}")
    for obs in e.get("observations", []):
        print(f"      - {obs}")

print(f"
관계 {len(relations)}개")
for r in relations:
    print(f"  {r['from']} --[{r['relationType']}]--> {r['to']}")

> 💡 Step 2 의 (1) 셀에 다른 문장을 넣어 몇 번 더 기억시킨 뒤 이 셀을 다시 실행해 보세요.
> 같은 사람에 대한 사실이 **새 개체로 생기는지, 기존 개체에 덧붙는지** 확인할 수 있습니다.

### 실습 2. 도구 필터 빼고 검색시키기

Step 3 과 같은 질문을 **필터 없이** 실행합니다. 서버의 도구가 전부 에이전트에게 보입니다.

**관찰할 것**
- 에이전트에게 보이는 도구가 몇 개로 늘었나요?
- `tavily_search` 말고 다른 도구(`tavily_extract`, `tavily_crawl` 등)를 골랐나요?
- 답의 품질이나 걸린 시간이 달라졌나요? (trace 에서 토큰 수도 비교해 보세요)

In [ ]:
system_prompt = "당신은 웹서치를 하고 요약하는 임무를 수행합니다."
model = "gpt-5.4-mini"
question = "사이버보안 취약점 정보 포탈에 접속하여 가장 최근에 발생한 보안 취약점에 대해서 요약해 주세요"  # ✏️ 질문을 바꿔 보세요

use_filter = False  # ✏️ True 로 바꾸면 Step 3 과 같은 조건

async with MCPServerStdio(params=tavily_params, client_session_timeout_seconds=60,
                          tool_filter=search_only if use_filter else None) as mcp_server:
    agent = Agent(name="agent", instructions=system_prompt, model=model, mcp_servers=[mcp_server])
    visible = await agent.get_mcp_tools(agents.RunContextWrapper(context=None))
    print(f"에이전트에게 보이는 도구 {len(visible)}개: {[t.name for t in visible]}")

    with trace(f"search filter={use_filter}"):
        result = await Runner.run(agent, question)

show_tool_calls(result)
display(Markdown(result.final_output))

### 실습 3. 저장 없이 꺼내기만 하기

Step 4 의 "꺼내 쓰기" 와 같은 코드인데, **아무것도 저장하지 않은 새 컬렉션**을 가리킵니다.

**관찰할 것**
- `qdrant-find` 는 무엇을 돌려주나요?
- 에이전트는 "모른다"고 하나요, 아니면 **자기 지식으로 그럴듯하게 지어내나요?**
- `COLLECTION_NAME` 을 `"knowledge"` 로 되돌리면 답이 어떻게 달라지나요?

In [ ]:
empty_collection = "empty_test"  # ✏️ "knowledge" 로 바꿔서 비교해 보세요

empty_params = {
    **vectorstore_params,
    "env": {**vectorstore_params["env"], "COLLECTION_NAME": empty_collection},
}

system_prompt = """당신의 임무는 주어진 주제에 대해 웹서치를 하고 취득한 정보를 
               벡터 데이터베이스에 저장하는 것입니다. 사용자가 저장된 정보에 대해서 질문하면
               답변해 주세요"""

async with MCPServerStdio(params=empty_params, client_session_timeout_seconds=120) as vector_server:
    agent = Agent(name="researcher", instructions=system_prompt, model=model, mcp_servers=[vector_server])
    with trace(f"retrieve from {empty_collection}"):
        result = await Runner.run(agent, "인공지능을 이용한 최근 보안 공격은 어떤 내용이 있는지 벡터 데이터베이스는 내용에서 알려주세요")

show_tool_calls(result)
display(Markdown(result.final_output))

> 💡 지어낸 답이 나왔다면 `system_prompt` 에 *"벡터 데이터베이스에 없는 내용은 모른다고 답하세요"* 를 추가하고 다시 실행해 보세요.

### 실습 4. 네 서버를 한 에이전트에

서버 네 개를 동시에 띄우려면 `async with` 를 네 번 겹쳐야 합니다.
`AsyncExitStack` 을 쓰면 **목록으로** 열고, 셀이 끝날 때 한꺼번에 닫습니다.

**관찰할 것**
- 도구가 모두 몇 개인가요? 서버별로는 몇 개씩인가요?
- 질문과 상관없는 서버의 도구를 부르지는 않나요?
- 같은 질문을 Step 2~5 의 **서버 하나짜리** 에이전트에게 했을 때와 결과·시간·토큰이 어떻게 다른가요?
- `use_filter` 를 `True` 로 바꾸면 도구 수와 선택이 어떻게 달라지나요?

In [ ]:
from contextlib import AsyncExitStack

use_filter = False  # ✏️ True 면 Tavily 에 search_only 필터 적용

all_params = {
    "memory": (memory_params, None),
    "tavily": (tavily_params, search_only if use_filter else None),
    "qdrant": (vectorstore_params, None),
}
if os.getenv("MASSIVE_API_KEY"):
    all_params["massive"] = (market_params, None)
else:
    print("MASSIVE_API_KEY 가 없어 massive 서버는 빼고 진행합니다.")

system_prompt = """당신은 여러 도구를 가진 비서입니다. 
사용자에 대한 사실은 기억 도구에, 조사한 자료는 벡터 데이터베이스에 저장하고, 
필요하면 웹 검색과 시장 데이터 도구를 사용하세요."""

question = "내 이름은 홍길동입니다. 내가 강의하는 주제와 관련된 최근 뉴스를 하나 찾아서 요약해 주세요"  # ✏️ 질문을 바꿔 보세요

async with AsyncExitStack() as stack:
    servers = []
    for name, (params, tool_filter) in all_params.items():
        server = await stack.enter_async_context(
            MCPServerStdio(params=params, client_session_timeout_seconds=120,
                           tool_filter=tool_filter, name=name))
        servers.append(server)

    agent = Agent(name="all-in-one", instructions=system_prompt, model=model, mcp_servers=servers)

    ctx = agents.RunContextWrapper(context=None)
    total = 0
    for server in servers:
        names = [t.name for t in await server.list_tools(ctx, agent)]
        total += len(names)
        print(f"[{server.name}] {len(names)}개: {names}")
    print(f"
에이전트에게 보이는 도구 합계: {total}개
")

    with trace("all servers"):
        result = await Runner.run(agent, question, max_turns=20)

show_tool_calls(result)
display(Markdown(result.final_output))